# Create mock normalization table

## Import

In [1]:
import os
import duckdb

from sindex.metrics.mocknormalization import (
    load_topics_csv_to_duckdb,
    create_mock_topic_norm_factors_table,
    validate_mock_table_nondecreasing,
)

from sindex.metrics.normalization import get_topic_year_norm_factors

## Paths

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
topics_path = os.path.join(parent_dir, "input", "openalex_topics")
TOPICS_CSV = os.path.join(topics_path, "topics.csv")
norm_path = os.path.join(parent_dir, "input", "mock_norm")
DB_PATH = os.path.join(norm_path, "mock_norm.duckdb")

## Initialize

In [3]:
con = duckdb.connect(DB_PATH)

## Load topics from CSV to DuckDB

In [4]:
load_topics_csv_to_duckdb(
    con,
    csv_path=TOPICS_CSV,
    topics_table="openalex_topics",
)

display(con.execute("SELECT COUNT(*) AS n_topics FROM openalex_topics").df())
display(con.execute("SELECT * FROM openalex_topics LIMIT 3").df())


,n_topics
0,4516


,topic_id,topic_name,subfield_name,field_name,domain_name,topic_id_short
0,https://openalex.org/T10346,Magnetic confinement fusion research,Nuclear and High Energy Physics,Physics and Astronomy,Physical Sciences,T10346
1,https://openalex.org/T12157,Geochemistry and Geologic Mapping,Artificial Intelligence,Computer Science,Physical Sciences,T12157
2,https://openalex.org/T10451,Mycorrhizal Fungi and Plant Interactions,Plant Science,Agricultural and Biological Sciences,Life Sciences,T10451


## Create mock normalization table

In [6]:
create_mock_topic_norm_factors_table(
    con,
    topics_table="openalex_topics",
    topic_id_col="topic_id",      
    topic_name_col="topic_name",
    out_table="topic_norm_factors_mock",
    year_start=2010,
    year_end=2025,
    # cfg={...}  # optional overrides
)

# Optional sanity check: should be 0 violations
violations = validate_mock_table_nondecreasing(con, table="topic_norm_factors_mock")
print("Monotonicity violations:", violations)

display(con.execute("SELECT COUNT(*) AS n_rows FROM topic_norm_factors_mock").df())
display(con.execute("""
    SELECT * FROM topic_norm_factors_mock
    WHERE topic_id='ALL'
    ORDER BY year
    LIMIT 8
""").df())

Monotonicity violations: 0


,n_rows
0,76789


,topic_id,topic_name,year,ft_median,ctw_median,mtw_median,n_datasets_f,n_datasets_c,n_datasets_m,is_general,is_mock
0,ALL,All datasets,-1,0.610,3.925,2.814,381,381,381,True,True
1,ALL,All datasets,2010,0.578,2.404,1.732,171,171,171,True,True
2,ALL,All datasets,2011,0.591,2.404,1.732,230,230,230,True,True
3,ALL,All datasets,2012,0.591,2.903,2.056,274,274,274,True,True
4,ALL,All datasets,2013,0.591,2.949,2.292,279,279,279,True,True
5,ALL,All datasets,2014,0.601,3.388,2.292,279,279,279,True,True
6,ALL,All datasets,2015,0.607,3.388,2.452,279,279,279,True,True
7,ALL,All datasets,2016,0.607,3.388,2.452,316,316,316,True,True


## Test queries

In [4]:
topic = "https://openalex.org/T10346"
year = 2025

norm = get_topic_year_norm_factors(
    con,
    topic_id=topic,
    year=year,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.693,
 'CTw': 6.227,
 'MTw': 2.344,
 'topic_id_used': 'https://openalex.org/T10346',
 'year_used': 2025,
 'topic_id_requested': 'https://openalex.org/T10346',
 'year_requested': 2025,
 'used_year_clamp': False}

In [5]:
topic = "https://openalex.org/T10346"

norm = get_topic_year_norm_factors(
    con,
    topic_id=topic,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.6615,
 'CTw': 4.4395,
 'MTw': 1.493,
 'topic_id_used': 'https://openalex.org/T10346',
 'year_used': -1,
 'topic_id_requested': 'https://openalex.org/T10346',
 'year_requested': None,
 'used_year_clamp': False}

In [6]:
topic = "https://opensdfT10346"

norm = get_topic_year_norm_factors(
    con,
    topic_id=topic,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.61,
 'CTw': 3.925,
 'MTw': 2.814,
 'topic_id_used': 'ALL',
 'year_used': -1,
 'topic_id_requested': 'https://opensdfT10346',
 'year_requested': None,
 'used_year_clamp': False}

In [6]:
norm = get_topic_year_norm_factors(
    con,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.61,
 'CTw': 3.925,
 'MTw': 2.814,
 'topic_id_used': 'ALL',
 'year_used': -1,
 'topic_id_requested': None,
 'year_requested': None,
 'used_year_clamp': False}

In [7]:
topic = "https://openalex.org/T10346"
year = 2028

norm = get_topic_year_norm_factors(
    con,
    topic_id=topic,
    year = year,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.693,
 'CTw': 6.227,
 'MTw': 2.344,
 'topic_id_used': 'https://openalex.org/T10346',
 'year_used': 2025,
 'topic_id_requested': 'https://openalex.org/T10346',
 'year_requested': 2028,
 'used_year_clamp': True}

In [8]:
year = 2005

norm = get_topic_year_norm_factors(
    con,
    year = year,
    table="topic_norm_factors_mock",
)

display(norm)

{'FT': 0.578,
 'CTw': 2.404,
 'MTw': 1.732,
 'topic_id_used': 'ALL',
 'year_used': 2010,
 'topic_id_requested': None,
 'year_requested': 2005,
 'used_year_clamp': True}